EDA
Exploratory data analysis — distributions, nulls, cardinalities, outliers, and date ranges across all tables

Data quality issues — identify at least 3 problems in the raw data and explain how you would handle them, justifying your 
approach

Star schema proposal — propose the dimensional model you will build in Phase 2; draw it or describe it clearly, and justify your grain and dimension choices

In [82]:
import pandas as pd #importar la biblioteca pandas para manipulación de datos
from pathlib import Path #para manejar rutas de archivos de manera más eficiente
DATA_PATH = Path(r"C:/Users/ahlam/data_technical_tests/data/raw")

In [ ]:
tables = {
    "clients": pd.read_csv(DATA_PATH / "raw_clients.csv"),
    "projects": pd.read_csv(DATA_PATH / "raw_projects.csv"),
    "campaigns": pd.read_csv(DATA_PATH / "raw_campaigns.csv"),
    "questions": pd.read_csv(DATA_PATH / "raw_questions.csv"),
    "workers": pd.read_csv(DATA_PATH / "raw_workers.csv"),
    "pos": pd.read_csv(DATA_PATH / "raw_pos.csv"),
    "routes": pd.read_csv(DATA_PATH / "raw_routes.csv"),
    "route_employee": pd.read_csv(DATA_PATH / "raw_route_employee.csv"),
    "visits": pd.read_csv(DATA_PATH / "raw_visits.csv"),
    "responses": pd.read_csv(DATA_PATH / "raw_responses.csv")
}

"""Crea un diccionario llamado "tables" que contiene los nombres de las tablas como claves y 
los DataFrames correspondientes como valores. Esto permite acceder fácilmente a cada tabla por su nombre 
y realizar análisis exploratorio de datos (EDA) en cada una de ellas."""

In [ ]:
""" Con esta funcion revisa de cada tabla la distribución, nulos, cardinalidad, outliers y rangos de fechas   
De esta forma, con una sola función, para obtener una visión general rápida de cada tabla y detectar posibles 
problemas de calidad de datos o inconsistencias en las fechas. Y no tener que repetir el mismo código para cada tabla
y asi poder analizar y dar conclusiones de cada tabla de manera más eficiente."""


def eda_table(table_name): 
    df = tables[table_name]
    
    print(f"\n==== {table_name.upper()} ====\n") 

    # 1. Tamaño

    # imprime el número de filas y columnas de la tabla utilizando el atributo shape del DataFrame.
    print(f"Filas: {df.shape[0]}")
    print(f"Columnas: {df.shape[1]}")

    # 2. Nulos, tipos y cardinalidad

    # creamos un nuevo DataFrame llamado info_df que contiene información sobre los tipos de datos, valores nulos, 
    # porcentaje de nulos y cardinalidad de cada columna en el DataFrame original df. 
    # Luego, imprime este nuevo DataFrame para mostrar la estructura de la tabla.
    info_df = pd.DataFrame({
        'Tipo_Dato': df.dtypes,
        'Valores_Nulos': df.isnull().sum(),
        '%_Nulos': (df.isnull().sum() / len(df) * 100).round(2),
        'Cardinalidad': df.nunique()
    })

    print("\n--- ESTRUCTURA ---")
    print(info_df)

    # 3. Duplicados

    print("\nDuplicados completos:", df.duplicated().sum()) 

    # para verificar si hay filas duplicadas en el DataFrame df utilizando el método duplicated() de pandas.
    #  y luego sumar el resultado para obtener el número total de filas duplicadas.

    # 4. Date ranges 

    print("\n--- DATE RANGES ---")
    date_cols = df.columns[df.columns.str.contains("date|created|updated", case=False)]
    # busca columnas que contengan palabras clave relacionadas con fechas, como "date", "created" o "updated".

    if len(date_cols) > 0:
        # Fuerza astype(str) ya que las columnas de fecha pueden contener valores no estándar o nulos que causan 
        # problemas al calcular el mínimo y máximo como en la tabla viits que tiene (str) y valores nulos reales (float / NaN).

        print(df[date_cols].astype(str).agg(['min', 'max']).T) #nos mostrara el rango de fechas para cada columna 
        # detectada, mostrando la fecha mínima y máxima presente en cada una de ellas.
    else:
        print("No date columns detected")
    #con este condicional, evitamos que el código se rompa si no se detectan columnas de fecha, y en su lugar, 
    # se muestra un mensaje indicando que no se han encontrado columnas de fecha.

    # 5. Outliers / estadística numérica 
        
    print("\n--- NUMERIC SUMMARY (OUTLIERS CHECK) ---")
    num_df = df.select_dtypes(include="number") #seleccionar solo las columnas numéricas del DataFrame df 
    #con el método select_dtypes(), y luego imprimimos un resumen estadístico de estas columnas utilizando el método describe() de pandas

    if num_df.shape[1] > 0:
        print(num_df.describe().T)
    else:
        print("No numeric columns detected")
    #si detecta columnas numéricas, muestra un resumen estadístico sino, muestra un mensaje "no se han encontrado columnas numéricas"
   
    # 6. Vista rápida de datos

    print("\n--- FIRST & LAST ROWS ---")
    print(pd.concat([df.head(3), df.tail(3)]))
    # muestra las primeras 3 filas y las últimas 3 filas del DataFrame y lo muestra en conjunto con concat 

    return df 

In [85]:
df = eda_table("campaigns") #llamamos a la función eda_table con el nombre de la tabla "campaigns" para realizar
# el análisis exploratorio de esta tabla.


==== CAMPAIGNS ====

Filas: 72
Columnas: 16

--- ESTRUCTURA ---
                       Tipo_Dato  Valores_Nulos  %_Nulos  Cardinalidad
campaign_id                int64              0     0.00            72
campaign_code             object              2     2.78            70
campaign_name             object              0     0.00            59
project_id                 int64              0     0.00            28
project_code              object              7     9.72            26
client_code               object              0     0.00            14
campaign_start_date       object              0     0.00            62
campaign_end_date         object              0     0.00            64
is_active                 object              5     6.94             2
campaign_state_id          int64              0     0.00             5
campaign_state            object             13    18.06             4
total_visits_planned       int64              0     0.00            68
total_pos_pl

- Tabla CAMPAIGNS

La tabla contiene información de campañas asociadas a proyectos y clientes, incluyendo planificación operativa, estado.
lo que sugiere relaciones con otras tablas que deberán validarse en la fase de modelado.

Insights: 

No se detectan duplicados y campaign_id parece ser una primary key.
Existen inconsistencias en formatos de fecha y tipos de datos que deberán corregirse en la fase de limpieza(campaign_start_date / campaign_end_date). 
Los mayores porcentajes de nulos se concentran en las variables (campaign_state y visit_duration_minutes), se deberia valorar su impacto

Posibles acciones para Phase 2
Las fechas se deberia cambiar formato a datetime 
Valorar impacto de los nulos



In [86]:
df = eda_table("clients")


==== CLIENTS ====

Filas: 15
Columnas: 7

--- ESTRUCTURA ---
               Tipo_Dato  Valores_Nulos  %_Nulos  Cardinalidad
client_id          int64              0      0.0            14
client_name       object              0      0.0            14
company_id         int64              0      0.0             1
country           object              0      0.0             3
sector            object              0      0.0             3
created_at        object              0      0.0            13
updated_at_sys    object              0      0.0            14

Duplicados completos: 1

--- DATE RANGES ---
                       min         max
created_at      2018-05-23  2021-11-30
updated_at_sys  2019-04-14  2023-08-25

--- NUMERIC SUMMARY (OUTLIERS CHECK) ---
            count      mean       std  min  25%  50%   75%   max
client_id    15.0  7.066667  4.366539  1.0  3.5  7.0  10.5  14.0
company_id   15.0  1.000000  0.000000  1.0  1.0  1.0   1.0   1.0

--- FIRST & LAST ROWS ---
    cli

- Tabla CLIENTS

La tabla contiene información maestra de clientes, incluyendo datos de identificación, sector, país y fechas de creación y actualización.

Insights

- Se detecta 1 registro duplicado completo, que deberá eliminarse durante la fase de limpieza para preservar la unicidad de los clientes.
- No existen valores nulos en ninguna columna, lo que indica un alto nivel de completitud de los datos.
- Se observan inconsistencias en la columna country (spain, SPAIN, España), por lo que será necesario estandarizar los valores categóricos.
- client_id parece ser primary key, aunque existe un registro duplicado.
- company_id presenta una única categoría (cardinalidad = 1), podría tener un papel solo referencial.
- Las columnas de fecha (created_at, updated_at_sys) están almacenadas como object y deberán convertirse a formato datetime. Y parece que si consitencia en las fechas.

Posibles acciones para Phase 2
Duplicado eliminar
Columnas country estandarizar
Las fechas se deberia cambiar formato a datetime 


In [87]:
df = eda_table("pos")


==== POS ====

Filas: 250
Columnas: 12

--- ESTRUCTURA ---
                               Tipo_Dato  Valores_Nulos  %_Nulos  Cardinalidad
intervention_point_id              int64              0      0.0           250
intervention_point_code           object             14      5.6           236
intervention_point_name           object              6      2.4           244
intervention_point_address        object             15      6.0           234
intervention_point_province       object             14      5.6            59
intervention_point_locality       object             26     10.4            50
intervention_point_postal_code   float64             26     10.4           224
intervention_point_latitude      float64             15      6.0           235
intervention_point_longitude     float64              7      2.8           243
intervention_point_is_active      object             10      4.0             2
created_at                        object              0      0.0       

- Tabla POS (Points of Sale)

La tabla contiene información de los puntos de venta o intervención, incluyendo datos de ubicación, identificación, coordenadas geográficas y estado de actividad.

Insights

- No se detectan registros duplicados y intervention_point_id parece actuar como clave primaria de la tabla.
- Existen valores nulos en varios campos de localización (locality, postal_code, address, province), que podrían afectar análisis geográficos.
- Las coordenadas geográficas (latitude y longitude) presentan una alta completitud, lo que permite localizar los puntos.
- intervention_point_is_active está almacenado como object en lugar de un tipo booleano (True, False) y contiene algunos valores nulos, por lo que requerirá revisión durante la limpieza.
- Las columnas de fecha (created_at, updated_at) están almacenadas como texto y deberán convertirse a formato datetime.
- La alta cardinalidad de campos como nombre, dirección y código indica que cada registro representa un punto de venta individual.

Posibles acciones para Phase 2

Estandarización de tipos de datos (datetime y booleanos).
Tratamiento de valores nulos en atributos de localización.
Validación de consistencia geográfica entre provincia, localidad, código postal y coordenadas.


In [88]:
df = eda_table("projects")


==== PROJECTS ====

Filas: 28
Columnas: 8

--- ESTRUCTURA ---
                   Tipo_Dato  Valores_Nulos  %_Nulos  Cardinalidad
project_id             int64              0     0.00            28
project_name          object              0     0.00            28
project_code          object              2     7.14            26
client_id              int64              0     0.00            14
client_code           object              0     0.00            14
created_at            object              0     0.00            27
updated_at_sys        object              0     0.00            27
project_exportable      bool              0     0.00             2

Duplicados completos: 0

--- DATE RANGES ---
                       min         max
created_at      2024-01-04  2025-05-26
updated_at_sys  2024-05-26  2025-12-23

--- NUMERIC SUMMARY (OUTLIERS CHECK) ---
            count       mean       std  min   25%   50%    75%   max
project_id   28.0  14.500000  8.225975  1.0  7.75  14.5  21.

- Tabla PROJECTS

La tabla contiene información de proyectos asociados a clientes, incluyendo identificadores, códigos de negocio, fechas de creación y una marca de exportación.

Insights

- No se detectan registros duplicados y project_id parece actuar como clave primaria de la tabla.
- La calidad de los datos es alta, con ausencia total de valores nulos salvo en project_code, que presenta 2 registros faltantes (7.14%).
- Existe una relación clara con la tabla de clientes mediante client_id y client_code, lo que la convierte en una tabla intermedia clave dentro del modelo relacional.
- project_exportable ya está correctamente almacenada como variable booleana.
- Las columnas de fecha (created_at y updated_at_sys) están almacenadas como texto y deberán convertirse a formato datetime.
- La cardinalidad de client_id (14 valores para 28 proyectos) indica que un mismo cliente puede estar asociado a varios proyectos, sugiriendo una relación uno-a-muchos.

Posibles acciones para Phase 2

Conversión de campos de fecha a tipo datetime.
Revisión e imputación de los valores faltantes en project_code.
Validación de la integridad referencial entre proyectos y clientes.
Definición de project_id como clave primaria y client_id como clave foránea dentro del modelo de datos.

In [89]:
df = eda_table("questions")


==== QUESTIONS ====

Filas: 648
Columnas: 12

--- ESTRUCTURA ---
                        Tipo_Dato  Valores_Nulos  %_Nulos  Cardinalidad
question_id                 int64              0     0.00           648
campaign_id                 int64              0     0.00            72
campaign_code              object              0     0.00            72
question_code              object             46     7.10           602
question_name              object             18     2.78            26
question_type              object             28     4.32             4
question_category          object             54     8.33             4
question_order            float64             23     3.55            14
question_is_highlighted      bool              0     0.00             2
image_associated             bool              0     0.00             2
created_at                 object              0     0.00            55
updated_at_sys             object              0     0.00           20

Tabla QUESTIONS

La tabla parece contener las preguntas definidas para cada campaña, incluyendo su tipo, categoría, orden de aparición y algunos atributos de configuración.

Insights

- No se detectan registros duplicados y question_id parece actuar como clave primaria de la tabla.
- Existe una relación clara con la tabla CAMPAIGNS mediante campaign_id y campaign_code, indicando una relación de uno a muchos (una campaña puede contener múltiples preguntas).
Los valores nulos se concentran principalmente en question_code (7.1%), question_category (8.3%) y question_type (4.3%), afectando atributos descriptivos relevantes para la clasificación de preguntas.
- question_name presenta una cardinalidad muy baja (26 valores únicos para 648 registros), lo que sugiere reutilización de preguntas entre diferentes campañas.
- Las variables question_is_highlighted e image_associated están correctas como booleanas y no presentan valores.
- question_order presenta algunos valores nulos y deberá revisarse, ya que podría ser un atributo relevante para reconstruir el orden del cuestionario.
- Las columnas temporales (created_at, updated_at_sys) están almacenadas como texto y deberán convertirse a formato datetime.

Posibles acciones para Phase 2

Conversión de fechas a tipo datetime.
Revisión y tratamiento de valores nulos en question_code, question_type, question_category y question_order.
Validación de la integridad referencial entre preguntas y campañas.
Comprobación de la unicidad de question_code y análisis de la reutilización de preguntas entre campañas.

In [90]:
df = eda_table("responses")


==== RESPONSES ====

Filas: 10192
Columnas: 10

--- ESTRUCTURA ---
                        Tipo_Dato  Valores_Nulos  %_Nulos  Cardinalidad
answer_id                   int64              0     0.00         10192
visit_id                    int64              0     0.00          1747
question_id               float64            288     2.83           564
campaign_code              object              0     0.00            64
intervention_point_code    object            581     5.70           248
question_type              object            509     4.99             4
answer                     object           1025    10.06           128
expected_answer            object           6851    67.22             2
created_at                 object              0     0.00           144
updated_at_sys             object              0     0.00           130

Duplicados completos: 0

--- DATE RANGES ---
                       min         max
created_at      2025-12-27  2026-05-19
updated_at_sys  

- Tabla RESPONSES

La tabla contiene las respuestas registradas a nivel de visita, vinculando preguntas, campañas y puntos de intervención, lo que la convierte en una tabla de hechos central del modelo.

Insights

- No se detectan registros duplicados y answer_id actúa como identificador único de cada respuesta. PK
Existe una alta cardinalidad en visit_id y answer_id, lo que sugiere respuestas por visita.
- Se observan valores nulos en varias columnas, especialmente en answer (10.1%) y intervention_point_code (5.7%), lo que puede afectar el análisis de resultados.
- expected_answer presenta un alto porcentaje de valores nulos (67.2%), puede que esta variable solo está definida para ciertos tipos de preguntas.
- question_id presenta algunos valores nulos (2.8%) y tipo float, lo que podría indicar problemas de integridad referencial.
- Las variables created_at y updated_at_sys están almacenadas como texto y deberán convertirse a formato datetime.


Posibles acciones para Phase 2

Validación de integridad referencial entre question_id, campaign_code y intervention_point_code.
Tratamiento de valores nulos en answer, especialmente para análisis cualitativo.
Revisión del campo expected_answer, ya que su alto nivel de missingness sugiere que no es aplicable a todos los registros.
Conversión de fechas a formato datetime.
Revisión del tipo de dato de question_id para asegurar consistencia con la tabla QUESTIONS.

In [91]:
df = eda_table("route_employee")


==== ROUTE_EMPLOYEE ====

Filas: 482
Columnas: 8

--- ESTRUCTURA ---
                  Tipo_Dato  Valores_Nulos  %_Nulos  Cardinalidad
route_employee_id     int64              0     0.00           482
route_id              int64              0     0.00           241
employee_id           int64              0     0.00            34
main_employee          bool              0     0.00             2
ip_percentage       float64             14     2.90           172
created_at           object              0     0.00           128
updated_at           object              0     0.00           122
deleted_at           object            457    94.81            24

Duplicados completos: 0

--- DATE RANGES ---
                   min         max
created_at  2025-10-02  2026-04-20
updated_at  2025-11-03  2026-05-19

--- NUMERIC SUMMARY (OUTLIERS CHECK) ---
                   count        mean         std   min     25%     50%  \
route_employee_id  482.0  241.500000  139.285678   1.0  121.25  241.5

- Tabla ROUTE_EMPLOYEE

La tabla representa la asignación de empleados a rutas, incluyendo su rol dentro de la ruta.

Insights

- No se detectan registros duplicados y route_employee_id actúa como PK de la tabla.
- La estructura refleja una relación muchos-a-muchos entre route_id y employee_id, lo que indica una tabla intermedia
- ip_percentage presenta pocos valores nulos (2.9%) hay un 75% cercanos al 100%, podria sugerir niveles de participación por empleado en cada ruta.
- deleted_at presenta un alto porcentaje de nulos (94.8%), lo que probablemente indica un campo donde la mayoría de registros siguen activos.
- Las columnas de fecha (created_at, updated_at) están en formato object y deberán convertirse a datetime.

Posibles acciones para Phase 2

Conversión de fechas a formato datetime.
Revisión de la consistencia de ip_percentage (posibles reglas de negocio:100%).
Identificación de empleados principales por ruta usando main_employee.

In [92]:
df = eda_table("routes")


==== ROUTES ====

Filas: 241
Columnas: 12

--- ESTRUCTURA ---
                 Tipo_Dato  Valores_Nulos  %_Nulos  Cardinalidad
route_id             int64              0     0.00           241
route_code          object             12     4.98           229
route_name          object              8     3.32           233
campaign_id          int64              0     0.00            72
campaign_code       object              0     0.00            72
route_start_date    object              0     0.00           134
route_end_date      object              0     0.00           168
route_status        object             78    32.37             3
delegation_code     object             27    11.20            15
recall_mail_sent      bool              0     0.00             2
created_at          object              0     0.00           128
updated_at_sys      object              0     0.00           122

Duplicados completos: 0

--- DATE RANGES ---
                         min         max
route

- Tabla ROUTES

La tabla contiene información de rutas asociadas a campañas, incluyendo planificación temporal, estado operativo y asignación a delegaciones.

Insights

- No se detectan registros duplicados y route_id actúa como PK de la tabla.
- Existe una relación clara con la tabla CAMPAIGNS mediante campaign_id y campaign_code. 
- Se observan valores nulos en route_status (32.4%), lo que sugiere una inconsistente del estado operativo de las rutas.
- También hay valores faltantes en delegation_code (11.2%), Se detectan inconsistencias en la cobertura de route_code y route_name, con algunos valores faltantes que podrian afectar a las rutas.
- Las variables de fecha (route_start_date, route_end_date, created_at, updated_at_sys) están almacenadas como object y deberán convertirse a formato datetime.
- La cardinalidad de route_status es muy baja (3 valores), lo que indica una variable categórica de estado bien definida, aunque incompleta.

Posibles acciones para Phase 2

Conversión de fechas a tipo datetime.
Tratamiento de valores nulos en route_status y delegation_code.
Validación de la integridad de la relación campaña → rutas.
Revisión de la consistencia de estados de ruta (definir catálogo de estados estándar).

In [93]:
df = eda_table("visits")


==== VISITS ====

Filas: 1762
Columnas: 15

--- ESTRUCTURA ---
                        Tipo_Dato  Valores_Nulos  %_Nulos  Cardinalidad
visit_id                    int64              0     0.00          1747
campaign_id                 int64              0     0.00            64
campaign_code              object              0     0.00            64
project_code               object              0     0.00            32
intervention_point_id       int64              0     0.00           248
intervention_point_code    object             95     5.39           248
route_id                  float64            156     8.85           270
route_code                 object            134     7.60           270
visit_date                 object             37     2.10           242
visit_time                 object            180    10.22            48
visit_status               object             71     4.03             4
visit_type                 object            666    37.80             3


- Tabla VISITS

La tabla contiene el registro de visitas realizadas, actuando como una tabla de hechos central que conecta campañas, proyectos, rutas y puntos de intervención.

Insights

- Se detectan 15 registros duplicados, lo que deberá corregirse en la fase de limpieza para evitar sobreconteo de visitas.
- Existe una alta conectividad con otras tablas mediante campaign_id, project_code, route_id e intervention_point_id, lo que confirma su rol como tabla central de eventos.
- Se observan valores nulos relevantes en varias columnas operativas:
visit_type (37.8%) → alto nivel de incertidumbre en la clasificación de visitas.
visit_time (10.2%) → impacto en análisis temporal detallado.
route_id y route_code (~7–9%) → posibles visitas no asignadas a ruta.
- Se detectan inconsistencias de formato en visit_date (mezcla de YYYY-MM-DD y DD/MM/YYYY), lo que requiere estandarización.
- route_id presenta valores atípicos muy elevados (máx. 99970), posibles errores de asignación. Ademas que debria estar com int.
- is_client_billable contiene valores nulos y está almacenado como object, por lo que requiere conversión a booleano.
- Las columnas de fecha (visit_date, created_at, updated_at_sys) están en formato object y deberán convertirse a datetime.

Posibles acciones para Phase 2

Eliminación o revisar los 15 registros duplicados.
Normalización de formatos de fecha.
Revisión crítica de route_id por posibles errores de integridad referencial.
Tratamiento de valores nulos en visit_type y variables de planificación.
Conversión de is_client_billable a tipo booleano.
Validación de integridad entre visitas, rutas y campañas.

In [94]:
df = eda_table("workers")


==== WORKERS ====

Filas: 47
Columnas: 9

--- ESTRUCTURA ---
                          Tipo_Dato  Valores_Nulos  %_Nulos  Cardinalidad
employee_id                   int64              0     0.00            45
employee_first_name          object              0     0.00            14
employee_active_status       object              3     6.38             2
employee_hire_date           object              2     4.26            43
employee_address_province    object              1     2.13            33
employee_contract_type       object             14    29.79             3
company_id                    int64              0     0.00             1
created_at                   object              0     0.00            45
updated_at_sys               object              0     0.00            45

Duplicados completos: 2

--- DATE RANGES ---
                           min         max
employee_hire_date  2018-05-08         nan
created_at          2018-05-08  2025-05-23
updated_at_sys      202

- Tabla WORKERS

La tabla contiene información de empleados de la empresa, incluyendo datos personales, estado laboral, tipo de contrato y metadatos temporales.

Insights

- Se detectan 2 registros duplicados completos, que deberán eliminarse durante la fase de limpieza para asegurar la unicidad de los empleados.
- La calidad general de los datos es buena, aunque existen valores nulos relevantes en employee_contract_type (29.8%), lo que puede afectar análisis de segmentación laboral.
- También se observan nulos menores en employee_hire_date (4.3%) y employee_active_status (6.4%), que deberán revisarse por coherencia con el estado del empleado.
- employee_active_status debería ser una variable booleana, aunque actualmente está almacenada como object.
- employee_hire_date y otras columnas de fecha están en formato object y deberán convertirse a datetime.
- company_id presenta cardinalidad 1, lo que indica que todos los empleados pertenecen a la misma empresa (posible redundancia en el modelo).
- Se observan inconsistencias en employee_address_province (mezcla de formatos), lo que requerirá estandarización.

Posibles acciones para Phase 2

Eliminación de duplicados.
Conversión de fechas a datetime.
Estandarización de variables categóricas (provincias, tipo de contrato).
Tratamiento de valores nulos en employee_contract_type.
Conversión de employee_active_status a tipo booleano.
Validación de consistencia de datos laborales (activo vs fechas de contratación).

# Data Quality Issues Identified (Raw Data)

A partir del análisis exploratorio de todas las tablas, se identifican los siguientes problemas principales en los datos crudos:

1. Inconsistencias en formatos de fecha
- Problema: En múltiples tablas (CAMPAIGNS, VISITS, ROUTES, WORKERS, etc.) las fechas están almacenadas como object y además presentan formatos mixtos:
YYYY-MM-DD
DD/MM/YYYY

- Solución:
Convertir todas las fechas a formato estándar datetime
Usar pd.to_datetime(..., errors="coerce")

Justificación:
Esto puede darnos errores en filtros temporales, joins por fecha y en el análisis, agregaciones y modelado de las fechas e imposibilidad de construir features temporales fiables.

2. Problemas de integridad referencial entre tablas
Se detectan posibles inconsistencias en claves especialmente en VISITS y RESPONSES, donde algunos identificadores como route_id y question_id presentan valores nulos, tipos incorrectos o posibles valores no existentes en las tablas de dimensión correspondientes.
Esto puede generar problemas en los joins entre tablas y afectar la consistencia del modelo relacional.

** revisar code: client_code, project_code

- Solución: Revisar PK que sean únicos y no duplicados
Validación de foreign keys contra tablas maestras
Identificación de IDs faltantes
Corrección de tipos (int vs float)
Eliminación o imputación de registros inconsistentes según reglas de negocio

3. Tipos de datos incorrectos o inconsistentes
Booleanos almacenados como object (is_active, is_client_billable)
IDs con tipos inconsistentes en algunas tablas
Errores en filtros y agregaciones
Dificultad para análisis estadístico correcto
Riesgo de bugs en modelado

- Solución:
bool para flags
datetime para fechas
int para IDs
Validación de schema antes del modelado
